In [0]:
%run "/Workspace/Repos/shoyofromconcrete@gmail.com/MultiChannelDataPipeLine/config"



In [0]:
import logging
import time
import traceback
import functools
import builtins
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ------------------------
# Logger Setup 
# ------------------------
logger = logging.getLogger(CONFIG["logging"]["logger_name"])
logger.setLevel(CONFIG["logging"]["log_level"])

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)

# ------------------------
# Set catalog/schema
# ------------------------
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schemas']['silver']}")

In [0]:
def read_bronze(source_name):

    table_name = CONFIG["tables"]["bronze"][source_name]

    return spark.table(
        f"{CONFIG['catalog']}.{CONFIG['schemas']['bronze']}.{table_name}"
    )

In [0]:
def standardize_df(df, source):

    mapping = CONFIG["COLUMN_MAPPINGS"][source]
    standard_cols = CONFIG["standard_cols"]
    common_cols = CONFIG["common_cols"]

    renamed_cols = [
        col(src).alias(dest)
        for src, dest in mapping.items()
        if src in df.columns
    ]

    common = [col(c) for c in common_cols if c in df.columns]

    df_std = df.select(*renamed_cols, *common)

    for c in standard_cols:
        if c not in df_std.columns:
            df_std = df_std.withColumn(c, lit(None))

    df_std = df_std.withColumn("source", lit(source))

    return df_std.select(*(standard_cols + common_cols))

In [0]:
def union_all(dfs):

    return functools.reduce(
        lambda df1, df2: df1.unionByName(df2, True),
        dfs.values()
    )

In [0]:
def parse_mixed_timestamp(col_name):

    c = col(col_name).cast("string")

    return (
        when(c.isNull() | (c == ""), None)
        .when(c.rlike(r"^\d{4}-\d{2}-\d{2}T"), to_timestamp(c))
        .when(c.rlike(r"^\d{4}-\d{2}-\d{2}$"), to_timestamp(c, "yyyy-MM-dd"))
        .when(c.rlike(r"^\d{2}/\d{2}/\d{2}$"), to_timestamp(c, "dd/MM/yy"))
        .when(c.rlike(r"^\d{10}$"), from_unixtime(c.cast("long")))
        .when(c.rlike(r"^\d{13}$"), from_unixtime((c.cast("long") / 1000)))
        .otherwise(None)
    )

In [0]:
def cast_columns(df, schema_dict):

    for c_name, dtype in schema_dict.items():

        if c_name not in df.columns:
            continue

        current_type = dict(df.dtypes)[c_name]

        if dtype == "timestamp":
            if current_type != "timestamp":
                df = df.withColumn(c_name, parse_mixed_timestamp(c_name))

        elif dtype == "date":
            if current_type != "date":
                df = df.withColumn(
                    c_name,
                    to_date(parse_mixed_timestamp(c_name))
                )

        else:
            if current_type != dtype:
                df = df.withColumn(c_name, col(c_name).cast(dtype))

    return df

In [0]:
def clean_data(df):

    return (
        df
        .withColumn("discount", abs(col("discount").cast("double")))
    )

In [0]:
def write_silver(df):

    table_name = f"{CONFIG['catalog']}.{CONFIG['schemas']['silver']}.sales_silver"

    try:
        start_time = time.time()

        logger.info(f"[START] Silver write | table={table_name}")

        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("mergeSchema", "true")
              .saveAsTable(table_name)
        )

        duration = builtins.round(time.time() - start_time, 2)

        if CONFIG["logging"]["track_execution_time"]:
            logger.info(
                f"[END] Silver write | table={table_name} | {duration} sec"
            )

    except Exception:
        logger.error(f"[ERROR] Silver write failed | table={table_name}")
        logger.error(traceback.format_exc())
        raise

In [0]:
def run_silver():

    try:
        logger.info("[START] Silver Pipeline")

        sources = list(CONFIG["tables"]["bronze"].keys())

        # Step 1: Read
        bronze_dfs = {
            src: read_bronze(src)
            for src in sources
        }

        # Step 2: Standardize
        silver_dfs = {
            src: standardize_df(bronze_dfs[src], src)
            for src in sources
        }

        # Step 3: Union
        final_df = union_all(silver_dfs)

        # Step 4: Cast
        final_df = cast_columns(final_df, TARGET_SCHEMA)

        # Step 5: Clean
        final_df = clean_data(final_df)

        # Step 6: Write
        write_silver(final_df)

        logger.info("[END] Silver Pipeline SUCCESS")

    except Exception:
        logger.error("[FAILED] Silver Pipeline")
        logger.error(traceback.format_exc())
        raise

In [0]:
run_silver()

In [0]:
%sql
select * from multidatadumps.silver.sales_silver